<a href="https://colab.research.google.com/github/niranjansendilkumar11/Irish-weather-pipeline/blob/main/ca2_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import requests



In [ ]:
#API Configuration
from google.colab import userdata

try:
  API_KEY =userdata.get('OWM_API_KEY')
  print("API Key loaded successfully!")
except Exception:
  API_KEY = ""


API Key loaded successfully!


In [ ]:
IRISH_CITIES = [
    "Dublin,IE",
    "Cork,IE",
    "Galway,IE",
    "Limerick,IE",
    "Waterford,IE",
    "Swords,IE",
    "Sligo,IE",
    "Derry,IE",
    "Drogheda,IE",
    "Tallaght,IE"
]
print(f"Monitoring {len(IRISH_CITIES)} Irish cities:")
for city in IRISH_CITIES:
    print(f"{city.split(',')[0]}")


Monitoring 10 Irish cities:
Dublin
Cork
Galway
Limerick
Waterford
Swords
Sligo
Derry
Drogheda
Tallaght


In [ ]:
import time

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

def fetch_city_weather(city_name):
    params = {
        "q": city_name,
        "appid": API_KEY,
        "units": "metric"
    }
    try:
      response = requests.get(BASE_URL, params=params,timeout=10)
      if response.status_code == 200:
        data = response.json()
        return {
                "city":          data["name"],
                "country":       data["sys"]["country"],
                "latitude":      data["coord"]["lat"],
                "longitude":     data["coord"]["lon"],
                "temp_celsius":  data["main"]["temp"],
                "feels_like":    data["main"]["feels_like"],
                "temp_min":      data["main"]["temp_min"],
                "temp_max":      data["main"]["temp_max"],
                "humidity":      data["main"]["humidity"],
                "pressure":      data["main"]["pressure"],
                "visibility":    data.get("visibility", None),
                "wind_speed":    data["wind"]["speed"],
                "wind_degree":   data["wind"].get("deg", None),
                "weather_main":  data["weather"][0]["main"],
                "weather_desc":  data["weather"][0]["description"],
                "cloud_coverage":data["clouds"]["all"],
                "sunrise":       data["sys"]["sunrise"],
                "sunset":        data["sys"]["sunset"],
                "fetched_at":    datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            }
      else:
            print(f"Failed for {city_name} - Status: {response.status_code}")
            return None
    except requests.exceptions.Timeout:
        print(f"Timeout for {city_name}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error for {city_name}: {e}")
        return None


In [ ]:
result = fetch_city_weather ("Dublin,IE")
print(result)

{'city': 'Dublin', 'country': 'IE', 'latitude': 53.344, 'longitude': -6.2672, 'temp_celsius': 9.03, 'feels_like': 6.29, 'temp_min': 9.03, 'temp_max': 9.03, 'humidity': 72, 'pressure': 1023, 'visibility': 10000, 'wind_speed': 5.22, 'wind_degree': 257, 'weather_main': 'Clouds', 'weather_desc': 'overcast clouds', 'cloud_coverage': 96, 'sunrise': 1774505603, 'sunset': 1774550881, 'fetched_at': '2026-03-26 11:37:49'}


In [ ]:
#Section 4 - Data Acquisition
# This section fetches live weather data for all 10 Irish cities
def fetch_all_cities(cities):
  all_records = []
  for city in cities:
    print(f"Fetching {city.split(',')[0]}...")
    record = fetch_city_weather(city)
    if record:
      all_records.append(record)
    time.sleep(1)
  df_raw = pd.DataFrame(all_records)
  print(f"Done Fetched {len(all_records)} cities")
  return df_raw

df_raw = fetch_all_cities(IRISH_CITIES)


Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetching Swords...
Fetching Sligo...
Fetching Derry...
Fetching Drogheda...
Fetching Tallaght...
Done Fetched 10 cities


In [ ]:
# Displaying the raw  data we fetched
df_raw.head(10)

,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,visibility,wind_speed,wind_degree,weather_main,weather_desc,cloud_coverage,sunrise,sunset,fetched_at
0,Dublin,IE,53.3440,-6.2672,9.03,6.29,9.03,9.03,72,1023,10000,5.22,257,Clouds,overcast clouds,96,1774505603,1774550881,2026-03-26 11:37:49
1,Cork,IE,51.8980,-8.4706,8.10,6.05,8.10,8.10,82,1025,10000,3.30,214,Clouds,overcast clouds,98,1774506178,1774551363,2026-03-26 11:37:50
2,Galway,IE,53.2719,-9.0489,7.49,3.33,7.49,7.49,90,1022,10000,8.27,209,Clouds,overcast clouds,100,1774506272,1774551547,2026-03-26 11:37:52
3,Limerick,IE,52.6647,-8.6231,7.57,5.12,7.57,7.57,84,1024,10000,3.80,191,Clouds,overcast clouds,99,1774506190,1774551424,2026-03-26 11:37:53
4,Waterford,IE,52.2583,-7.1119,9.50,7.40,9.50,9.50,69,1024,10000,3.97,250,Clouds,broken clouds,73,1774505841,1774551048,2026-03-26 11:37:54
5,Swords,IE,53.4597,-6.2181,8.92,6.02,8.92,8.92,73,1023,10000,5.57,255,Clouds,overcast clouds,99,1774505587,1774550873,2026-03-26 11:37:55
6,Sligo,IE,54.2697,-8.4694,6.73,3.55,6.73,6.73,92,1020,10000,4.88,178,Clouds,overcast clouds,98,1774506099,1774551442,2026-03-26 11:37:56
7,Derry,IE,51.5867,-9.0503,7.72,4.13,7.72,7.72,87,1025,10000,6.62,234,Clouds,overcast clouds,100,1774506326,1774551493,2026-03-26 11:37:57
8,Drogheda,IE,53.7189,-6.3478,8.30,5.13,8.30,8.30,74,1023,10000,5.88,251,Clouds,overcast clouds,100,1774505610,1774550913,2026-03-26 11:37:58
9,Tallaght,IE,53.2859,-6.3734,8.43,5.55,8.43,8.43,74,1023,10000,5.19,251,Clouds,overcast clouds,91,1774505630,1774550904,2026-03-26 11:37:59


In [ ]:
#section 5- Transformations
# This section cleans and enriches the raw weather data

def categorise_temperature (temp_celsius):
  if temp_celsius < 0:
    return "Freezing"
  elif temp_celsius < 8:
        return "Cold"
  elif temp_celsius < 14:
        return "Cool"
  elif temp_celsius < 19:
        return "Mild"
  else:
        return "Warm"

In [ ]:
def categorise_wind(wind_speed):
    if wind_speed < 3.0:
        return "Calm"
    elif wind_speed < 8.0:
        return "Breezy"
    elif wind_speed < 14.0:
        return "Windy"
    else:
        return "Storm"


In [ ]:
def categorise_humidity(humidity):
  if humidity < 40:
        return "Dry"
  elif humidity < 70:
        return "Comfortable"
  else:
        return "Humid"

In [ ]:
#calculating how many hours of daylight each city gets
def convert_unix_timestamp(unix_time):
    return datetime.fromtimestamp(unix_time).strftime('%H:%M:%S')

def calculate_daylight_hours(sunrise_unix, sunset_unix):
    duration_seconds = sunset_unix - sunrise_unix
    return round(duration_seconds / 3600, 2)

print(convert_unix_timestamp(1774333095))
print(calculate_daylight_hours(1774333095, 1774377864))

06:18:15
12.44


In [ ]:
# Applies all transformations to the raw dataframe and returns a cleaned enriched version
def transform_weather_data(df):
  df_transformed = df.copy()

  df_transformed["sunrise_time"] = df_transformed["sunrise"].apply(convert_unix_timestamp)
  df_transformed["sunset_time"] = df_transformed["sunset"].apply(convert_unix_timestamp)
  df_transformed["daylight_hours"] = df_transformed.apply(
        lambda row: calculate_daylight_hours(row["sunrise"], row["sunset"]), axis=1
    )
  df_transformed["temp_category"] = df_transformed["temp_celsius"].apply(categorise_temperature)
  df_transformed["wind_category"] = df_transformed["wind_speed"].apply(categorise_wind)
  df_transformed["humidity_category"] = df_transformed["humidity"].apply(categorise_humidity)
  df_transformed["visibility_km"] = (df_transformed["visibility"] / 1000).round(2)
  df_transformed.drop(columns=["sunrise", "sunset", "visibility"], inplace=True)

  return df_transformed

df_transformed = transform_weather_data(df_raw)
print("Transformations applied successfully!")
print(f"Columns now: {df_transformed.shape[1]}")
df_transformed.head(10)

Transformations applied successfully!
Columns now: 23


,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,...,weather_desc,cloud_coverage,fetched_at,sunrise_time,sunset_time,daylight_hours,temp_category,wind_category,humidity_category,visibility_km
0,Dublin,IE,53.3440,-6.2672,9.03,6.29,9.03,9.03,72,1023,...,overcast clouds,96,2026-03-26 11:37:49,06:13:23,18:48:01,12.58,Cool,Breezy,Humid,10.0
1,Cork,IE,51.8980,-8.4706,8.10,6.05,8.10,8.10,82,1025,...,overcast clouds,98,2026-03-26 11:37:50,06:22:58,18:56:03,12.55,Cool,Breezy,Humid,10.0
2,Galway,IE,53.2719,-9.0489,7.49,3.33,7.49,7.49,90,1022,...,overcast clouds,100,2026-03-26 11:37:52,06:24:32,18:59:07,12.58,Cold,Windy,Humid,10.0
3,Limerick,IE,52.6647,-8.6231,7.57,5.12,7.57,7.57,84,1024,...,overcast clouds,99,2026-03-26 11:37:53,06:23:10,18:57:04,12.56,Cold,Breezy,Humid,10.0
4,Waterford,IE,52.2583,-7.1119,9.50,7.40,9.50,9.50,69,1024,...,broken clouds,73,2026-03-26 11:37:54,06:17:21,18:50:48,12.56,Cool,Breezy,Comfortable,10.0
5,Swords,IE,53.4597,-6.2181,8.92,6.02,8.92,8.92,73,1023,...,overcast clouds,99,2026-03-26 11:37:55,06:13:07,18:47:53,12.58,Cool,Breezy,Humid,10.0
6,Sligo,IE,54.2697,-8.4694,6.73,3.55,6.73,6.73,92,1020,...,overcast clouds,98,2026-03-26 11:37:56,06:21:39,18:57:22,12.60,Cold,Breezy,Humid,10.0
7,Derry,IE,51.5867,-9.0503,7.72,4.13,7.72,7.72,87,1025,...,overcast clouds,100,2026-03-26 11:37:57,06:25:26,18:58:13,12.55,Cold,Breezy,Humid,10.0
8,Drogheda,IE,53.7189,-6.3478,8.30,5.13,8.30,8.30,74,1023,...,overcast clouds,100,2026-03-26 11:37:58,06:13:30,18:48:33,12.58,Cool,Breezy,Humid,10.0
9,Tallaght,IE,53.2859,-6.3734,8.43,5.55,8.43,8.43,74,1023,...,overcast clouds,91,2026-03-26 11:37:59,06:13:50,18:48:24,12.58,Cool,Breezy,Humid,10.0


In [ ]:
# Section 6 - Unit Testing
#Testing each transformation function to verify they return correct results
import unittest

class TestWeatherData(unittest.TestCase):

  def test_categorise_temperature(self):
    self.assertEqual(categorise_temperature(-5), "Freezing")
    self.assertEqual(categorise_temperature(4), "Cold")
    self.assertEqual(categorise_temperature(10), "Cool")
    self.assertEqual(categorise_temperature(16), "Mild")
    self.assertEqual(categorise_temperature(22), "Warm")

  def test_categorise_wind(self):
    self.assertEqual(categorise_wind(1.5), "Calm")
    self.assertEqual(categorise_wind(5.0), "Breezy")
    self.assertEqual(categorise_wind(10.0), "Windy")
    self.assertEqual(categorise_wind(18.0), "Storm")

  def test_categorise_humidity(self):
    self.assertEqual(categorise_humidity(30), "Dry")
    self.assertEqual(categorise_humidity(55), "Comfortable")
    self.assertEqual(categorise_humidity(85), "Humid")

  def test_calculate_daylight_hours(self):
    self.assertEqual(calculate_daylight_hours(1774333095, 1774377864), 12.44)

unittest.main(argv=[""], exit=False, verbosity=2)

test_calculate_daylight_hours (__main__.TestWeatherData.test_calculate_daylight_hours) ... ok
test_categorise_humidity (__main__.TestWeatherData.test_categorise_humidity) ... ok
test_categorise_temperature (__main__.TestWeatherData.test_categorise_temperature) ... ok
test_categorise_wind (__main__.TestWeatherData.test_categorise_wind) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.009s

OK


In [ ]:
# Section 7 - Integration Test
# Testing the full pipeline from data fetching to transformation

def test_full_pipeline():
    print("Running integration test...")

    #Fetching data for only one city
    test_data = fetch_city_weather("Dublin,IE")
    assert test_data is not None, "Fetch failed - no data returned"
    assert "temp_celsius" in test_data, "temp_celsius missing from fetched data"
    assert "humidity" in test_data, "humidity missing from fetched data"
    print("Data fetched successfully")

    test_df = pd.DataFrame([test_data])
    test_df_transformed = transform_weather_data(test_df)
    assert "temp_category" in test_df_transformed.columns, "temp_category missing from transformed data"
    assert "wind_category" in test_df_transformed.columns, "wind_category column missing"
    assert "sunrise_time" in test_df_transformed.columns, "sunrise_time column missing"
    print("Transformations applied successfully")

    assert test_df_transformed.shape[0] == 1, "Expected 1 row"
    assert test_df_transformed.shape[1] == 23, "Expected 23 columns"

    print("Integration test passed")

test_full_pipeline()



Running integration test...
Data fetched successfully
Transformations applied successfully
Integration test passed


In [ ]:
#Section 8 - Save data to CSV locally every hour in a CSV file

import os

def save_to_csv(df, filename="irish_weather_data.csv"):

  if os.path.exists(filename):
    df.to_csv(filename, mode='a', header=False, index=False)
    print(f"Data appended to {filename}")
  else:
     df.to_csv(filename, mode='w', header=True, index=False)
     print(f"New file created: {filename}")

  print(f"Rows saved: {len(df)}")

save_to_csv(df_transformed)




Data appended to irish_weather_data.csv
Rows saved: 10


In [ ]:
# Section 9 - Hourly Scheduler

def run_pipeline():
    print(f"Pipeline started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    df_raw = fetch_all_cities(IRISH_CITIES)

    df_transformed = transform_weather_data(df_raw)

    save_to_csv(df_transformed)

    print(f"Pipeline completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total rows in file so far:")
    df_check = pd.read_csv("irish_weather_data.csv")
    print(df_check.shape)

run_pipeline()

#running every hour
while True:
    print("Waiting 1 hour before next run...")
    time.sleep(3600)
    run_pipeline()




Pipeline started at: 2026-03-26 11:46:09
Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetching Swords...
Fetching Sligo...
Fetching Derry...
Fetching Drogheda...
Fetching Tallaght...
Done Fetched 10 cities
Data appended to irish_weather_data.csv
Rows saved: 10
Pipeline completed at: 2026-03-26 11:46:19
Total rows in file so far:
(40, 23)
Waiting 1 hour before next run...
Pipeline started at: 2026-03-26 11:47:19
Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetching Swords...
Fetching Sligo...
Fetching Derry...
Fetching Drogheda...
Fetching Tallaght...
Done Fetched 10 cities
Data appended to irish_weather_data.csv
Rows saved: 10
Pipeline completed at: 2026-03-26 11:47:29
Total rows in file so far:
(50, 23)
Waiting 1 hour before next run...
Pipeline started at: 2026-03-26 11:48:29
Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetchin

KeyboardInterrupt: 